# 🚗 เทรนโมเดลรวม CULane + Thai Road Lane (Joint Training)
## Model: DeepLabV3+ (ResNet50 + SSL)

สมุดงานนี้รวมชุดข้อมูล **CULane** และ **thai_road_lane** เข้าด้วยกัน:
- **Architecture:** `DeepLabV3Plus`
- **Encoder:** `resnet50` (`encoder_weights="ssl"`)
- **Train Set:** `ConcatDataset([Thai Train (749), CULane Train])`
- **Val Set:** วัดผลบน Val ทั้งสองชุด
- **Hyperparameters:** `EPOCHS = 30`, `BATCH_SIZE = 16`, `LR = 1e-4`
- **Auto-Resume:** มีระบบบันทึก Checkpoint ทุก Epoch หลุดแล้วรันต่อได้ทันที

In [1]:
# ตรวจสอบการติดตั้งแพ็กเกจ
!pip install -q opencv-python matplotlib numpy segmentation-models-pytorch tqdm pandas

In [2]:
import os
import sys
import csv
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  ใช้อุปกรณ์: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM ทั้งหมด: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🖥️  ใช้อุปกรณ์: cuda
   GPU: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition MIG 1g.24gb
   VRAM ทั้งหมด: 25.37 GB


In [3]:
def calculate_iou(pred_mask, true_mask, threshold=0.5):
    pred_bin = (pred_mask > threshold).bool()
    true_bin = (true_mask > 0.5).bool()
    intersection = (pred_bin & true_bin).sum().float()
    union = (pred_bin | true_bin).sum().float()
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return (intersection / union).item()


def calculate_precision_recall_f1(pred_mask, true_mask, threshold=0.5):
    pred_bin = (pred_mask > threshold).bool()
    true_bin = (true_mask > 0.5).bool()
    tp = (pred_bin & true_bin).sum().float()
    fp = (pred_bin & ~true_bin).sum().float()
    fn = (~pred_bin & true_bin).sum().float()

    precision = (tp / (tp + fp)).item() if (tp + fp) > 0 else 0.0
    recall = (tp / (tp + fn)).item() if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

In [4]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs.float()).view(-1)
        targets = targets.float().view(-1)
        intersection = (inputs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice


bce_criterion = nn.BCEWithLogitsLoss()
dice_criterion = DiceLoss()

In [5]:
IMG_SIZE = (448, 800)


class UniversalLaneDataset(Dataset):
    """Dataset รองรับทั้ง CULane และ Thai Road Lane"""
    def __init__(self, csv_file, img_dir, mask_dir, augment=False, domain_name=""):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.augment = augment
        self.domain_name = domain_name
        self.df = pd.read_csv(csv_file)
        self.samples = self.df["filename"].tolist()
        self.categories = self.df["category"].tolist() if "category" in self.df.columns else ["normal"] * len(self.samples)

        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname = self.samples[idx]
        img_p = os.path.join(self.img_dir, fname)
        mask_name = os.path.splitext(fname)[0] + ".png"
        mask_p = os.path.join(self.mask_dir, mask_name)

        img = cv2.imread(img_p)
        if img is None:
            raise FileNotFoundError(f"ไม่พบภาพ: {img_p}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if img.shape[:2] != IMG_SIZE:
            img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_LINEAR)

        mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"ไม่พบ mask: {mask_p}")
        if mask.shape[:2] != IMG_SIZE:
            mask = cv2.resize(mask, (IMG_SIZE[1], IMG_SIZE[0]), interpolation=cv2.INTER_NEAREST)

        if self.augment:
            if np.random.rand() > 0.5:
                img = np.fliplr(img).copy()
                mask = np.fliplr(mask).copy()
            if np.random.rand() > 0.5:
                factor = np.random.uniform(0.7, 1.3)
                img = np.clip(img * factor, 0, 255).astype(np.uint8)

        img_norm = (img.astype(np.float32) / 255.0 - self.mean) / self.std
        img_tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)

        return img_tensor, mask_tensor

In [6]:
def save_training_plot(history_train_loss, history_val_loss,
                       history_train_iou, history_val_iou,
                       history_train_recall, history_val_recall,
                       history_train_f1, history_val_f1,
                       graph_path):
    plt.figure(figsize=(14, 10))

    plt.subplot(2, 2, 1)
    plt.plot(history_train_loss, label="Train Loss (Joint)", color="red")
    plt.plot(history_val_loss, label="Val Loss (Thai)", color="orange")
    plt.title("Loss History", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc="upper right")

    plt.subplot(2, 2, 2)
    plt.plot(history_train_iou, label="Train IoU (Joint)", color="green")
    plt.plot(history_val_iou, label="Val IoU (Thai)", color="blue")
    plt.title("Accuracy (IoU) History", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.legend(loc="upper left")

    plt.subplot(2, 2, 3)
    plt.plot(history_train_recall, label="Train Recall (Joint)", color="purple")
    plt.plot(history_val_recall, label="Val Recall (Thai)", color="magenta")
    plt.title("Recall History", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.legend(loc="upper left")

    plt.subplot(2, 2, 4)
    plt.plot(history_train_f1, label="Train F1 (Joint)", color="teal")
    plt.plot(history_val_f1, label="Val F1 (Thai)", color="navy")
    plt.title("F1-score History", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("F1")
    plt.legend(loc="upper left")

    plt.tight_layout()
    os.makedirs(os.path.dirname(graph_path), exist_ok=True)
    plt.savefig(graph_path, dpi=110)
    plt.close()
    print(f"📈 บันทึกกราฟไว้ที่: {graph_path}")

## 🔧 CONFIG — ตั้งค่าเทรนแบบรวม (Joint Training)

In [7]:
# ==================== แก้ตรงนี้อย่างเดียว ====================
ARCH            = "Unet"
ENCODER_NAME    = "resnet50"
ENCODER_WEIGHTS = "imagenet"

ROOT_DIR        = ".." if os.path.exists("../thai_road_lane") else "."
THAI_ROOT       = os.path.join(ROOT_DIR, "thai_road_lane")
CULANE_ROOT     = "/home/jovyan/shared/donut/CarDashCam/CULane"

# สัดส่วนภาพ CULane ที่จะนำมารวม (None = เอา CULane ทั้งหมด 61,182 ภาพ)
# แนะนำ: ใส่ตัวเลข เช่น 749 หรือ 1500 หากต้องการให้บาลานซ์กับถนนไทย (749 ภาพ) และเทรนเร็ว
# หากต้องการใช้ทั้ง CULane 61,182 ภาพเต็มๆ ให้ใส่ CULANE_TRAIN_SAMPLES = None
CULANE_TRAIN_SAMPLES = None

EPOCHS          = 30
BATCH_SIZE      = 16
LEARNING_RATE   = 1e-4
NUM_WORKERS     = 4
USE_AMP         = True

# ตั้งชื่อ TAG อัตโนมัติตาม ARCH, ENCODER, WEIGHTS ป้องกันไฟล์เซฟทับกัน
_weights_tag    = str(ENCODER_WEIGHTS).lower() if ENCODER_WEIGHTS else "scratch"
TAG             = f"joint_culane_thai_{ARCH.lower()}_{ENCODER_NAME}_{_weights_tag}"
# ================================================================

print(f"🚀 โมเดล: {ARCH} + {ENCODER_NAME} (encoder_weights='{ENCODER_WEIGHTS}')")
print(f"   Model Tag:      {TAG}")
print(f"   Thai Dataset:   {THAI_ROOT} (พบ: {os.path.exists(THAI_ROOT)})")
print(f"   CULane Dataset: {CULANE_ROOT} (พบ: {os.path.exists(CULANE_ROOT)})")
culane_desc = "ทั้งหมด 61,182 ภาพ" if CULANE_TRAIN_SAMPLES is None else f"{CULANE_TRAIN_SAMPLES} ภาพ (Balanced Subset)"
print(f"   CULane Samples: {culane_desc}")
print(f"   Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LEARNING_RATE}")

🚀 โมเดล: Unet + resnet50 (encoder_weights='imagenet')
   Model Tag:      joint_culane_thai_unet_resnet50_imagenet
   Thai Dataset:   ../thai_road_lane (พบ: True)
   CULane Dataset: /home/jovyan/shared/donut/CarDashCam/CULane (พบ: True)
   CULane Samples: ทั้งหมด 61,182 ภาพ
   Epochs: 30 | Batch Size: 16 | LR: 0.0001


In [8]:
def build_model():
    model = getattr(smp, ARCH)(
        encoder_name=ENCODER_NAME,
        encoder_weights=ENCODER_WEIGHTS,
        in_channels=3,
        classes=1
    )
    return model.to(device)

model = build_model()
print(f"✅ สร้างโมเดลสำเร็จ: {ARCH} + {ENCODER_NAME} ({ENCODER_WEIGHTS})")

✅ สร้างโมเดลสำเร็จ: Unet + resnet50 (imagenet)


In [9]:
# โหลด Thai Dataset
thai_train_ds = UniversalLaneDataset(f"{THAI_ROOT}/train_list.csv", f"{THAI_ROOT}/images", f"{THAI_ROOT}/masks", augment=True, domain_name="Thai")
thai_val_ds   = UniversalLaneDataset(f"{THAI_ROOT}/val_list.csv", f"{THAI_ROOT}/images", f"{THAI_ROOT}/masks", augment=False, domain_name="Thai")

# โหลด CULane Dataset
culane_train_full = UniversalLaneDataset(f"{CULANE_ROOT}/train_list.csv", f"{CULANE_ROOT}/images", f"{CULANE_ROOT}/masks", augment=True, domain_name="CULane")
culane_val_full   = UniversalLaneDataset(f"{CULANE_ROOT}/val_list.csv", f"{CULANE_ROOT}/images", f"{CULANE_ROOT}/masks", augment=False, domain_name="CULane")

if CULANE_TRAIN_SAMPLES is not None and CULANE_TRAIN_SAMPLES < len(culane_train_full):
    indices = np.linspace(0, len(culane_train_full) - 1, CULANE_TRAIN_SAMPLES, dtype=int)
    culane_train_ds = Subset(culane_train_full, indices)
else:
    culane_train_ds = culane_train_full

# รวม Train Set เข้าด้วยกัน (Joint ConcatDataset)
joint_train_ds = ConcatDataset([thai_train_ds, culane_train_ds])

train_loader = DataLoader(joint_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader_thai = DataLoader(thai_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"📊 ขนาดข้อมูลเทรนรวม (Joint): {len(joint_train_ds):,} ภาพ (Thai {len(thai_train_ds):,} + CULane {len(culane_train_ds):,})")
print(f"   ขนาดข้อมูล Val ไทย: {len(thai_val_ds)} ภาพ")

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)

best_f1 = 0.0
best_epoch = 0
best_model_path  = os.path.join(ROOT_DIR, "models", f"{TAG}_best_f1.pth")
final_model_path = os.path.join(ROOT_DIR, "models", f"{TAG}_final.pth")
resume_ckpt_path = os.path.join(ROOT_DIR, "models", f"{TAG}_resume.pth")
csv_history_path = os.path.join(ROOT_DIR, "results", "thai", f"history_{TAG}.csv")
graph_path       = os.path.join(ROOT_DIR, "results", "thai", f"training_history_{TAG}.png")

history_train_loss, history_val_loss = [], []
history_train_iou,  history_val_iou  = [], []
history_train_rec,  history_val_rec  = [], []
history_train_f1,   history_val_f1   = [], []

start_epoch = 1

# ระบบ Auto-Resume
if os.path.exists(resume_ckpt_path):
    print(f"🔄 ตรวจพบ Resume Checkpoint: {resume_ckpt_path}")
    ckpt = torch.load(resume_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    if ckpt.get("optimizer_state_dict") and "param_groups" in ckpt["optimizer_state_dict"]:
        try:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        except Exception:
            pass
    if ckpt.get("scheduler_state_dict") and isinstance(ckpt["scheduler_state_dict"], dict) and len(ckpt["scheduler_state_dict"]) > 0:
        try:
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        except Exception:
            pass
    if ckpt.get("scaler_state_dict") and isinstance(ckpt["scaler_state_dict"], dict) and len(ckpt["scaler_state_dict"]) > 0 and USE_AMP:
        try:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        except Exception:
            pass
    best_f1 = ckpt.get("best_f1", 0.0)
    best_epoch = ckpt.get("best_epoch", 0)
    history_train_loss = ckpt.get("history_train_loss", [])
    history_val_loss   = ckpt.get("history_val_loss", [])
    history_train_iou  = ckpt.get("history_train_iou", [])
    history_val_iou    = ckpt.get("history_val_iou", [])
    history_train_rec  = ckpt.get("history_train_rec", [])
    history_val_rec    = ckpt.get("history_val_rec", [])
    history_train_f1   = ckpt.get("history_train_f1", [])
    history_val_f1     = ckpt.get("history_val_f1", [])
    start_epoch = ckpt["epoch"] + 1
    print(f"✅ Auto-Resume สำเร็จ! กำลังเทรนต่อจาก Epoch {start_epoch}/{EPOCHS} (ไม่ต้องเริ่มใหม่)")

if start_epoch == 1:
    os.makedirs(os.path.dirname(csv_history_path), exist_ok=True)
    with open(csv_history_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "train_iou", "train_f1", "val_loss", "val_iou", "val_recall", "val_f1"])

if start_epoch > EPOCHS:
    print(f"🎉 โมเดลเทรนครบทั้งหมด {EPOCHS} Epochs แล้ว! ข้ามไปยังขั้นตอนประเมินผล")
else:
    print(f"\n🏁 กำลังรันการเทรนรวม Joint CULane + Thai (Epoch {start_epoch} -> {EPOCHS})...")
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()
        t_loss = t_iou = t_prec = t_rec = t_f1 = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS} [Train Joint]", leave=False)
        for images, masks in pbar:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                outputs = model(images)
                loss = bce_criterion(outputs, masks) + dice_criterion(outputs, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                preds = torch.sigmoid(outputs.float())
                iou = calculate_iou(preds, masks)
                p, r, f1 = calculate_precision_recall_f1(preds, masks)

            t_loss += loss.item()
            t_iou  += iou
            t_rec  += r
            t_f1   += f1
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "f1": f"{f1:.4f}"})

        n_t = len(train_loader)
        ep_t_loss, ep_t_iou, ep_t_rec, ep_t_f1 = t_loss/n_t, t_iou/n_t, t_rec/n_t, t_f1/n_t

        # Validation บนถนนไทย (เป้าหมายหลัก)
        model.eval()
        v_loss = v_iou = v_prec = v_rec = v_f1 = 0.0
        with torch.inference_mode():
            for images, masks in val_loader_thai:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=USE_AMP):
                    outputs = model(images)
                    loss = bce_criterion(outputs, masks) + dice_criterion(outputs, masks)

                preds = torch.sigmoid(outputs.float())
                iou = calculate_iou(preds, masks)
                p, r, f1 = calculate_precision_recall_f1(preds, masks)

                v_loss += loss.item()
                v_iou  += iou
                v_rec  += r
                v_f1   += f1

        n_v = len(val_loader_thai)
        ep_v_loss, ep_v_iou, ep_v_rec, ep_v_f1 = v_loss/n_v, v_iou/n_v, v_rec/n_v, v_f1/n_v

        scheduler.step(ep_v_f1)

        history_train_loss.append(ep_t_loss); history_val_loss.append(ep_v_loss)
        history_train_iou.append(ep_t_iou);   history_val_iou.append(ep_v_iou)
        history_train_rec.append(ep_t_rec);   history_val_rec.append(ep_v_rec)
        history_train_f1.append(ep_t_f1);     history_val_f1.append(ep_v_f1)

        with open(csv_history_path, "a", newline="") as f:
            csv.writer(f).writerow([epoch, ep_t_loss, ep_t_iou, ep_t_f1, ep_v_loss, ep_v_iou, ep_v_rec, ep_v_f1])

        saved_mark = ""
        if ep_v_f1 > best_f1:
            best_f1 = ep_v_f1
            best_epoch = epoch
            torch.save(model.state_dict(), best_model_path)
            saved_mark = f" 💾 [Best Val F1={best_f1:.4f}]"

        # บันทึก Resume Checkpoint
        resume_checkpoint_data = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_f1": best_f1,
            "best_epoch": best_epoch,
            "history_train_loss": history_train_loss,
            "history_val_loss": history_val_loss,
            "history_train_iou": history_train_iou,
            "history_val_iou": history_val_iou,
            "history_train_rec": history_train_rec,
            "history_val_rec": history_val_rec,
            "history_train_f1": history_train_f1,
            "history_val_f1": history_val_f1,
        }
        torch.save(resume_checkpoint_data, resume_ckpt_path)

        save_training_plot(history_train_loss, history_val_loss, history_train_iou, history_val_iou,
                           history_train_rec, history_val_rec, history_train_f1, history_val_f1, graph_path)

        print(f"[{epoch:02d}/{EPOCHS}] Train Loss: {ep_t_loss:.4f} | Val IoU: {ep_v_iou:.4f} | Val F1: {ep_v_f1:.4f}{saved_mark}")

    torch.save(model.state_dict(), final_model_path)
    print(f"\n🎉 การเทรนเสร็จสิ้น! 🏆 Best Epoch คือ {best_epoch} (Val F1={best_f1:.4f})")
    print(f"📁 Best Model:  {best_model_path}")
    print(f"📁 Final Model: {final_model_path}")
    print(f"📁 History CSV: {csv_history_path}")
    print(f"📈 History Plot:{graph_path}")

📊 ขนาดข้อมูลเทรนรวม (Joint): 61,931 ภาพ (Thai 749 + CULane 61,182)
   ขนาดข้อมูล Val ไทย: 97 ภาพ
🔄 ตรวจพบ Resume Checkpoint: ../models/joint_culane_thai_unet_resnet50_imagenet_resume.pth
✅ Auto-Resume สำเร็จ! กำลังเทรนต่อจาก Epoch 31/30 (ไม่ต้องเริ่มใหม่)
🎉 โมเดลเทรนครบทั้งหมด 30 Epochs แล้ว! ข้ามไปยังขั้นตอนประเมินผล


In [10]:
# วัดผลบนชุดทดสอบทางการถนนไทย (191 ภาพ)
eval_model = getattr(smp, ARCH)(encoder_name=ENCODER_NAME, encoder_weights=None, in_channels=3, classes=1)
eval_model.load_state_dict(torch.load(best_model_path, map_location="cpu"))
eval_model = eval_model.to(device).eval()

test_ds = UniversalLaneDataset(f"{THAI_ROOT}/test_list.csv", f"{THAI_ROOT}/images", f"{THAI_ROOT}/masks", augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

meta_test = pd.read_csv(f"{THAI_ROOT}/test_list.csv")
cats = meta_test["category"].tolist()

acc = {}
def slot(cat):
    return acc.setdefault(cat, {"tp": 0., "fp": 0., "fn": 0., "inter": 0., "uni": 0., "img": 0})

img_idx = 0
with torch.inference_mode():
    for images, masks in tqdm(test_loader, desc="Testing Thai Set"):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            outputs = eval_model(images)
        preds = (torch.sigmoid(outputs.float())[:, 0] > 0.5).cpu().numpy()
        gts = (masks[:, 0] > 0.5).numpy()

        for b in range(len(preds)):
            cat = cats[img_idx] if img_idx < len(cats) else "normal"
            img_idx += 1
            pb = preds[b]
            gb = gts[b]
            s = slot(cat)
            all_s = slot("ALL")
            tp = float(np.logical_and(pb, gb).sum())
            fp = float(np.logical_and(pb, ~gb).sum())
            fn = float(np.logical_and(~pb, gb).sum())
            inter = tp
            uni = float(np.logical_or(pb, gb).sum())
            for target in (s, all_s):
                target["tp"] += tp
                target["fp"] += fp
                target["fn"] += fn
                target["inter"] += inter
                target["uni"] += uni
                target["img"] += 1

print("\n📊 === ผลการทดสอบโมเดล Joint Training บนถนนไทย (Test 191 ภาพ) ===")
for cat, d in sorted(acc.items()):
    p = d["tp"] / (d["tp"] + d["fp"] + 1e-8)
    r = d["tp"] / (d["tp"] + d["fn"] + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    iou = d["inter"] / (d["uni"] + 1e-8)
    print(f"{cat:10s} (N={d['img']:3.0f}) | IoU: {iou:.4f} | Prec: {p:.4f} | Rec: {r:.4f} | F1: {f1:.4f}")

Testing Thai Set:   0%|          | 0/12 [00:00<?, ?it/s]


📊 === ผลการทดสอบโมเดล Joint Training บนถนนไทย (Test 191 ภาพ) ===
ALL        (N=191) | IoU: 0.4998 | Prec: 0.7042 | Rec: 0.6325 | F1: 0.6665
curve      (N= 17) | IoU: 0.3328 | Prec: 0.7509 | Rec: 0.3741 | F1: 0.4994
night      (N= 17) | IoU: 0.1851 | Prec: 0.4972 | Rec: 0.2277 | F1: 0.3124
normal     (N=157) | IoU: 0.5677 | Prec: 0.7133 | Rec: 0.7354 | F1: 0.7242
